In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!git clone https://github.com/shengjunzi/ImgFi.git

In [ ]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

class WidarDataset(Dataset):
    def __init__(self, data_dir):
        self.data_dir = data_dir
        # Logic to list all your Widar 3.0 files (e.g., using os.listdir or glob)
        self.file_paths = [...] 
        self.labels = [...] # Extract labels based on the Widar 3.0 naming convention

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        # 1. Load the raw CSI file (Widar usually uses .dat or .csv)
        raw_csi = self.load_csi_file(self.file_paths[idx])
        
        # 2. Extract Amplitude: The paper requires magnitude variation
        # If raw_csi is complex numbers, amplitude = np.abs(raw_csi)
        amplitude_csi = np.abs(raw_csi)
        
        # 3. Apply the Imaging Transformation (See Phase 2)
        csi_image = generate_recurrence_plot(amplitude_csi)
        
        # Convert to PyTorch tensors
        image_tensor = torch.tensor(csi_image, dtype=torch.float32).unsqueeze(0) # Add channel dimension
        label_tensor = torch.tensor(self.labels[idx], dtype=torch.long)
        
        return image_tensor, label_tensor

In [ ]:
from scipy.spatial.distance import cdist

def generate_recurrence_plot(time_series_data):
    """
    Converts 1D CSI amplitude sequence into a 2D Recurrence Plot image.
    """
    # Ensure data is 2D for cdist (N, 1)
    time_series_reshaped = time_series_data.reshape(-1, 1)
    
    # Calculate absolute differences between all timestamps
    # cdist computes the distance between each pair of the two collections of inputs
    rp_matrix = cdist(time_series_reshaped, time_series_reshaped, metric='euclidean')
    
    # Optional: Normalize the matrix to 0-1 range for the CNN
    rp_matrix = (rp_matrix - np.min(rp_matrix)) / (np.max(rp_matrix) - np.min(rp_matrix) + 1e-8)
    
    return rp_matrix

In [ ]:
import os
import glob
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist

def generate_recurrence_plot(time_series_data):
    """Convert 1D CSI amplitude sequence into a 2D Recurrence Plot."""
    ts = time_series_data.reshape(-1, 1)
    rp = cdist(ts, ts, metric='euclidean')
    rp = (rp - rp.min()) / (rp.max() - rp.min() + 1e-8)
    return rp

# 1. Automatically find all .mat files in the Kaggle input directory
print("Scanning for .mat files...")
test_files = sorted(glob.glob('/kaggle/input/**/*.mat', recursive=True))

if not test_files:
    print("Hold up - couldn't find the .mat files. Check your data_dir path!")
    print("Listing directory structure to help debug:")
    for root, dirs, files in os.walk('/kaggle/input/'):
        print(f"  {root}")
        if files:
            print(f"    Sample files: {files[:3]}")
else:
    target_file = test_files[0]
    print(f"Target locked: Loading file -> {target_file}")
    
    # 2. Load the raw CSI data
    mat_data = sio.loadmat(target_file)
    
    # Find the data key dynamically
    data_keys = [k for k in mat_data.keys() if not k.startswith('__')]
    raw_csi = mat_data[data_keys[0]] 
    
    # Grab the first 500 data points of the amplitude
    amplitude_csi = np.abs(raw_csi.flatten()[:500]) 
    
    # 3. Pass it through your Recurrence Plot function
    csi_image = generate_recurrence_plot(amplitude_csi)
    
    # 4. Visualize the output
    plt.figure(figsize=(8, 8))
    plt.imshow(csi_image, cmap='jet', origin='lower')
    plt.title('CSI Recurrence Plot (ImgFi Transformation)')
    plt.colorbar(label='Amplitude Distance')
    plt.tight_layout()
    plt.show()


In [ ]:
import os
import glob
import numpy as np
import scipy.io as sio 
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist

# 1. The Recurrence Plot Function (Included here so it runs in one shot)
def generate_recurrence_plot(time_series_data):
    time_series_reshaped = time_series_data.reshape(-1, 1)
    rp_matrix = cdist(time_series_reshaped, time_series_reshaped, metric='euclidean')
    rp_matrix = (rp_matrix - np.min(rp_matrix)) / (np.max(rp_matrix) - np.min(rp_matrix) + 1e-8)
    return rp_matrix

# 2. Hunt down the files recursively
print("Scouting Kaggle directories for .mat files...")
all_mat_files = glob.glob('/kaggle/input/**/*.mat', recursive=True)

if not all_mat_files:
    print("Still empty. Let's print out the exact folder structure so we can see what's going on:")
    for root, dirs, files in os.walk('/kaggle/input/'):
        print(f"Directory: {root}")
        if files:
            print(f"  Sample Files: {files[:3]}")
else:
    # 3. Lock onto the first target
    target_file = all_mat_files[0]
    print(f"\nTarget locked! Found {len(all_mat_files)} files.")
    print(f"Extracting CSI from: {target_file}")
    
    # 4. Load the raw CSI data
    mat_data = sio.loadmat(target_file)
    data_keys = [k for k in mat_data.keys() if not k.startswith('__')]
    raw_csi = mat_data[data_keys[0]] 
    
    # Grab a slice of the amplitude for the 1D to 2D test
    amplitude_csi = np.abs(raw_csi.flatten()[:500]) 
    
    # 5. Transform
    csi_image = generate_recurrence_plot(amplitude_csi)
    
    # 6. Visualize
    plt.figure(figsize=(8, 8))
    plt.imshow(csi_image, cmap='magma', origin='lower') # Swapped to magma for a darker, cinematic aesthetic
    plt.title('CSI Recurrence Plot (ImgFi Transformation)')
    plt.colorbar(label='Amplitude Distance')
    plt.tight_layout()
    plt.show()

In [ ]:
import os
import glob
import numpy as np
import scipy.io as sio 
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist

# 1. The Recurrence Plot Function
def generate_recurrence_plot(time_series_data):
    time_series_reshaped = time_series_data.reshape(-1, 1)
    rp_matrix = cdist(time_series_reshaped, time_series_reshaped, metric='euclidean')
    rp_matrix = (rp_matrix - np.min(rp_matrix)) / (np.max(rp_matrix) - np.min(rp_matrix) + 1e-8)
    return rp_matrix

# 2. Find files recursively
print("Scouting Kaggle directories for .mat files...")
all_mat_files = glob.glob('/kaggle/input/**/*.mat', recursive=True)

if not all_mat_files:
    print("No .mat files found. Check your dataset configuration.")
else:
    target_file = all_mat_files[0]
    print(f"\nTarget locked! Found {len(all_mat_files)} files.")
    print(f"Inspecting file: {target_file}\n" + "-"*40)
    
    # 3. Safe Loading and Inspection
    mat_data = sio.loadmat(target_file)
    
    # Print keys to find out exactly what the data variable is named
    all_keys = list(mat_data.keys())
    print("Available keys inside this .mat file:", all_keys)
    
    # Filter out metadata keys starting with double underscores
    data_keys = [k for k in all_keys if not k.startswith('__')]
    print("Filtered potential data keys:", data_keys)
    
    if not data_keys:
        print("Error: No data keys found after filtering metadata.")
    else:
        # Use the first valid data key found dynamically
        chosen_key = data_keys[0]
        print(f"Extracting data using key: '{chosen_key}'")
        
        raw_csi = mat_data[chosen_key]
        print(f"Raw data shape extracted: {raw_csi.shape}")
        
        # 4. Flatten and take a 500-point slice for the image conversion test
        amplitude_csi = np.abs(raw_csi.flatten()[:500]) 
        
        # 5. Transform to image
        csi_image = generate_recurrence_plot(amplitude_csi)
        
        # 6. Visualize
        plt.figure(figsize=(8, 8))
        plt.imshow(csi_image, cmap='magma', origin='lower')
        plt.title(f'CSI Recurrence Plot (Key: {chosen_key})')
        plt.colorbar(label='Amplitude Distance')
        plt.tight_layout()
        plt.show()

In [ ]:
import os
import glob
import numpy as np
import scipy.io as sio
import cv2 
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from scipy.spatial.distance import cdist

# ---------------------------------------------------------
# 1. THE DATASET BUILDER
# ---------------------------------------------------------
class WidarImgFiDataset(Dataset):
    def __init__(self, file_paths, image_size=(224, 224)):
        self.file_paths = file_paths
        self.image_size = image_size

    def __len__(self):
        return len(self.file_paths)
    
    def extract_label(self, filepath):
        """
        Widar3.0 appends the label to the end of the train filename.
        Example: 'train-ENV1CNT10-2.mat' -> Label is 2.
        """
        base_name = os.path.basename(filepath)
        try:
            # Isolate the number after the last hyphen and drop '.mat'
            label_str = base_name.split('-')[-1].replace('.mat', '')
            label_id = int(label_str) 
            # PyTorch expects labels to start at 0 (so 1-6 becomes 0-5)
            return label_id - 1 
        except:
            return 0 

    def __getitem__(self, idx):
        mat_data = sio.loadmat(self.file_paths[idx])
        
        # Dynamically find the data key (avoids KeyError)
        data_keys = [k for k in mat_data.keys() if not k.startswith('__')]
        raw_csi = mat_data[data_keys[0]]
        
        # Slice: 1st subcarrier, 1st TX, 1st RX
        csi_stream = raw_csi[:, 0, 0, 0]
        amplitude_csi = np.abs(csi_stream)
        
        # Recurrence Plot Transformation
        time_series = amplitude_csi.reshape(-1, 1)
        rp_matrix = cdist(time_series, time_series, metric='euclidean')
        rp_matrix = (rp_matrix - np.min(rp_matrix)) / (np.max(rp_matrix) - np.min(rp_matrix) + 1e-8)
        
        # Resize to standard 224x224
        rp_image_resized = cv2.resize(rp_matrix, self.image_size, interpolation=cv2.INTER_LINEAR)
        
        image_tensor = torch.tensor(rp_image_resized, dtype=torch.float32).unsqueeze(0)
        label_tensor = torch.tensor(self.extract_label(self.file_paths[idx]), dtype=torch.long)
        
        return image_tensor, label_tensor

# ---------------------------------------------------------
# 2. THE ImgFi CNN ARCHITECTURE
# ---------------------------------------------------------
class ImgFi_CNN(nn.Module):
    def __init__(self, num_classes=6):
        super(ImgFi_CNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.adaptive_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.adaptive_pool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

# ---------------------------------------------------------
# 3. THE TRAINING ENGINE
# ---------------------------------------------------------
print("Scouting for full training dataset...")
# Grab ONLY the training files (Kaggle test files don't have labels appended)
all_train_files = glob.glob('/kaggle/input/**/train/**/*.mat', recursive=True)
print(f"Locked onto {len(all_train_files)} training files.")

dataset = WidarImgFiDataset(all_train_files)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=2) # Sped up dataloading

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ImgFi_CNN(num_classes=6).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training Loop
num_epochs = 20 
print("Initiating Full-Scale Training Sequence...")

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
        # Calculate training accuracy dynamically
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    epoch_loss = running_loss/len(train_loader)
    epoch_acc = 100 * correct / total
    print(f"Epoch [{epoch+1}/{num_epochs}] | Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.2f}%")

# Save the Weapon
model_save_path = 'ImgFi_Widar_Weights.pth'
torch.save(model.state_dict(), model_save_path)
print(f"\nTraining Complete. Model weights successfully saved to: {model_save_path}")

In [ ]:
import os
import glob
import random
import numpy as np
import scipy.io as sio
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from scipy.spatial.distance import cdist

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Representative low / mid / high subcarriers -> 3 image channels (paper: 3@224x224)
SUBCARRIERS = (5, 15, 25)
IMAGE_SIZE = (224, 224)

def generate_recurrence_plot(series):
    """1-D amplitude series -> normalized 2-D Recurrence Plot (pairwise distances)."""
    ts = series.reshape(-1, 1)
    rp = cdist(ts, ts, metric='euclidean')
    rp = (rp - rp.min()) / (rp.max() - rp.min() + 1e-8)
    return rp

# ---------------------------------------------------------
# 1. DATASET: 3-channel CSI Recurrence-Plot images
#    Each sample -> (3, 224, 224): RP of 3 subcarriers, amplitude averaged
#    over all 3x6 TX/RX antenna pairs (was previously 1 of 540 streams).
# ---------------------------------------------------------
class WidarImgFiDataset(Dataset):
    def __init__(self, file_paths, image_size=IMAGE_SIZE, subcarriers=SUBCARRIERS):
        self.file_paths = file_paths
        self.image_size = image_size
        self.subcarriers = subcarriers

    def __len__(self):
        return len(self.file_paths)

    def extract_label(self, filepath):
        """Widar train files end in '-<label>.mat' (e.g. train-ENV1CNT10-2.mat -> 2).
        Filename labels are 1-indexed; convert to 0-indexed for PyTorch."""
        base = os.path.basename(filepath)
        try:
            return int(base.split('-')[-1].replace('.mat', '')) - 1
        except ValueError:
            return 0

    def __getitem__(self, idx):
        mat = sio.loadmat(self.file_paths[idx])
        data_keys = [k for k in mat.keys() if not k.startswith('__')]
        raw_csi = mat[data_keys[0]]                 # (packets, 30, 3, 6)

        amp = np.abs(raw_csi)                        # amplitude (magnitude variation)
        amp = amp.mean(axis=(2, 3))                  # -> (packets, 30): avg over TX/RX

        channels = []
        for sc in self.subcarriers:
            rp = generate_recurrence_plot(amp[:, sc])
            rp = cv2.resize(rp, self.image_size, interpolation=cv2.INTER_LINEAR)
            channels.append(rp)
        image = np.stack(channels, axis=0)           # (3, 224, 224)

        image_tensor = torch.tensor(image, dtype=torch.float32)
        label_tensor = torch.tensor(self.extract_label(self.file_paths[idx]), dtype=torch.long)
        return image_tensor, label_tensor

# ---------------------------------------------------------
# 2. ImgFi CNN (4 conv layers) — 3-channel input
# ---------------------------------------------------------
class ImgFi_CNN(nn.Module):
    def __init__(self, num_classes=6, in_channels=3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512), nn.ReLU(), nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.adaptive_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(nn.Dropout(p=0.5), nn.Linear(512, num_classes))

    def forward(self, x):
        x = self.features(x)
        x = self.adaptive_pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

# ---------------------------------------------------------
# 3. FILE-LEVEL 70/15/15 SPLIT (split files, not images -> no leakage).
#    Only train-* files carry labels; test-* files are unlabeled, so the
#    held-out test set is carved from the labeled train files.
# ---------------------------------------------------------
print("Scouting for labeled training files...")
all_train_files = sorted(glob.glob('/kaggle/input/**/train/**/*.mat', recursive=True))
random.shuffle(all_train_files)

n = len(all_train_files)
n_train = int(0.70 * n)
n_val = int(0.15 * n)
train_files = all_train_files[:n_train]
val_files = all_train_files[n_train:n_train + n_val]
test_files = all_train_files[n_train + n_val:]
print(f"Total labeled files: {n} | train: {len(train_files)} | "
      f"val: {len(val_files)} | test: {len(test_files)}")

train_loader = DataLoader(WidarImgFiDataset(train_files), batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(WidarImgFiDataset(val_files), batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(WidarImgFiDataset(test_files), batch_size=32, shuffle=False, num_workers=2)

# ---------------------------------------------------------
# 4. TRAIN with validation-based best-model saving + LR scheduler
# ---------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Deploying model to: {device}")

model = ImgFi_CNN(num_classes=6, in_channels=3).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.8)  # paper decay rate 0.8

@torch.no_grad()
def evaluate_accuracy(loader):
    model.eval()
    correct = total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        _, predicted = torch.max(model(images), 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    return 100 * correct / total

num_epochs = 20
best_val_acc = 0.0
best_model_path = 'ImgFi_Widar_Weights.pth'
print("Initiating Full-Scale Training Sequence...")

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    scheduler.step()
    train_acc = 100 * correct / total
    val_acc = evaluate_accuracy(val_loader)

    tag = ""
    if val_acc > best_val_acc:                 # save BEST model on val acc (paper Algorithm 2)
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)
        tag = "  <-- best (saved)"

    print(f"Epoch [{epoch+1}/{num_epochs}] | Loss: {running_loss/len(train_loader):.4f} "
          f"| Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%{tag}")

print(f"\nTraining complete. Best val accuracy: {best_val_acc:.2f}% | weights: {best_model_path}")

# Reload best weights for the held-out test evaluation in the next cell
model.load_state_dict(torch.load(best_model_path))
print("Best model reloaded for evaluation.")


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import numpy as np
import torch
import matplotlib.pyplot as plt

print("Commencing Deep Analytics Evaluation on the HELD-OUT TEST set...")

# Evaluate on test_loader — files the model was NEVER trained or validated on.
model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        _, predicted = torch.max(model(images), 1)
        all_preds.extend(predicted.cpu().numpy())
        all_targets.extend(labels.cpu().numpy())

# Widar 3.0 gesture classes (label order matches the 0-indexed filename labels)
target_names = ['Push', 'Sweep', 'Clap', 'Slide', 'Draw-Z', 'Draw-N']

print("\n--- CLASSIFICATION REPORT (TEST SET) ---")
print(classification_report(all_targets, all_preds,
                            labels=np.arange(6), target_names=target_names,
                            zero_division=0))

# Confusion matrix (6x6 even if a class is missing from predictions)
cm = confusion_matrix(all_targets, all_preds, labels=np.arange(6))
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names)
plt.title('ImgFi Model Confusion Matrix (Test Set)', fontsize=16)
plt.xlabel('Predicted Gesture', fontsize=12)
plt.ylabel('True Gesture', fontsize=12)
plt.show()
